### np.corrcoef

In [5]:
import math

def get_shape(a):
    if not isinstance(a, list):
        return ()
    if len(a) == 0:
        return (0,)
    inner_shape = get_shape(a[0])
    for i in range(1, len(a)):
        if get_shape(a[i]) != inner_shape:
            raise ValueError(
                f"Jagged array: first row shape {inner_shape}, "
                f"but row {i} shape {get_shape(a[i])}"
            )
    return (len(a),) + inner_shape


def flatten(a):
    if not isinstance(a, list):
        return [a]
    result = []
    for elem in a:
        result.extend(flatten(elem))
    return result


def _mean(xs):
    return sum(xs) / len(xs)


def _std(xs):
    m = _mean(xs)
    return math.sqrt(sum((x - m) ** 2 for x in xs) / len(xs))


def _corr_1d(x, y):
    if len(x) != len(y):
        raise ValueError("x and y must have the same length")
    if len(x) == 0:
        raise ValueError("cannot compute correlation on empty arrays")

    mx = _mean(x)
    my = _mean(y)
    sx = _std(x)
    sy = _std(y)

    if sx == 0 or sy == 0:
        return float('nan')

    cov = sum((x[i] - mx) * (y[i] - my) for i in range(len(x))) / len(x)
    return cov / (sx * sy)


def _as_2d_vars(x, rowvar=True):
    shape = get_shape(x)
    if len(shape) == 0:
        return [[x]]
    if len(shape) == 1:
        return [x[:]]
    if len(shape) != 2:
        raise ValueError("Only 1-D or 2-D input is supported")

    if rowvar:
        return [row[:] for row in x]
    else:
        rows, cols = shape
        return [[x[i][j] for i in range(rows)] for j in range(cols)]


def np_corrcoef(x, y=None, rowvar=True):
    if y is not None:
        xv = _as_2d_vars(x, rowvar=rowvar)
        yv = _as_2d_vars(y, rowvar=rowvar)
        xshape = get_shape(xv)
        yshape = get_shape(yv)
        if xshape != yshape:
            raise ValueError("x and y must have the same shape")
        data = xv + yv
    else:
        data = _as_2d_vars(x, rowvar=rowvar)

    nvars = len(data)
    if nvars == 0:
        return []

    result = [[0.0] * nvars for _ in range(nvars)]
    for i in range(nvars):
        for j in range(nvars):
            result[i][j] = _corr_1d(data[i], data[j])
    return result

### test cases 

In [6]:
print(np_corrcoef([1, 2, 3, 4, 5]))
print(np_corrcoef([1, 2, 3, 4, 5], [2, 4, 6, 8, 10]))
print(np_corrcoef([1, 2, 3, 4, 5], [5, 4, 3, 2, 1]))
print(np_corrcoef([[1, 2, 3], [2, 4, 6]]))
print(np_corrcoef([[1, 2, 3], [2, 4, 6]], rowvar=False))
print(np_corrcoef([[1, 2, 3], [4, 5, 6]])) 

[[0.9999999999999998]]
[[0.9999999999999998, 0.9999999999999998], [0.9999999999999998, 0.9999999999999998]]
[[0.9999999999999998, -0.9999999999999998], [-0.9999999999999998, 0.9999999999999998]]
[[1.0, 1.0], [1.0, 1.0]]
[[1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0]]
[[1.0, 1.0], [1.0, 1.0]]


In [7]:
print(np_corrcoef([], [])) 

ValueError: cannot compute correlation on empty arrays

In [8]:
print(np_corrcoef([1, 2], [1])) 

ValueError: x and y must have the same shape

In [9]:
print(np_corrcoef([[1, 2], [3]], rowvar=True))

ValueError: Jagged array: first row shape (2,), but row 1 shape (1,)